In [42]:
import sys
sys.path.insert(0, "python")

import importlib
import pandas as pd
import pipeline

importlib.reload(pipeline)
raw = pipeline.load_raw("data/raw_results.csv")
r_md = pd.read_csv("r_outputs/match_data.csv")
print(raw.shape, r_md.shape)
raw.head()

(2052, 16) (1962, 17)


,Game,Date,Round,Home.Team,Home.Goals,Home.Behinds,Home.Points,Away.Team,Away.Goals,Away.Behinds,Away.Points,Venue,Margin,Season,Round.Type,Round.Number
0,14787,2016-03-24,R1,Richmond,14,8,92,Carlton,12,11,83,M.C.G.,9,2016,Regular,1
1,14788,2016-03-26,R1,Melbourne,12,8,80,GWS,10,18,78,M.C.G.,2,2016,Regular,1
2,14789,2016-03-26,R1,Gold Coast,17,19,121,Essendon,9,6,60,Carrara,61,2016,Regular,1
3,14790,2016-03-26,R1,North Melbourne,16,11,107,Adelaide,14,13,97,Docklands,10,2016,Regular,1
4,14791,2016-03-26,R1,Sydney,18,25,133,Collingwood,7,11,53,S.C.G.,80,2016,Regular,1


In [43]:
ha = pipeline.filter_regular(raw)
print(len(ha), len(r_md), len(ha) == len(r_md))

1962 1962 True


In [44]:
opening = pipeline.opening_round_seasons(ha)
print(opening)    # R printed: Opening Round seasons: 2024, 2025

[2024, 2025]


Another Check Here:

In [45]:
importlib.reload(pipeline)
ha = pipeline.filter_regular(raw)
print(len(ha), len(r_md), len(ha) == len(r_md))
print(pipeline.opening_round_seasons(ha))

1962 1962 True
[2024, 2025]


In [46]:
importlib.reload(pipeline)
ha = pipeline.filter_regular(raw)
md = pipeline.build_match_data(ha, pipeline.opening_round_seasons(ha))
md.to_csv("scratch_match_data.csv", index=False)
py = pd.read_csv("scratch_match_data.csv")

built = [c for c in r_md.columns if c in py.columns]
bad = [c for c in built if not (py[c].astype(str) == r_md[c].astype(str)).all()]
print("built:", len(built), "of", len(r_md.columns), "| mismatched:", bad)

built: 17 of 17 | mismatched: ['home_nickname', 'away_nickname']


In [47]:
diff = py["home_nickname"].astype(str) != r_md["home_nickname"].astype(str)
print(diff.sum(), "rows differ")
pd.concat(
    [py.loc[diff, ["season", "home_team", "home_nickname"]],
     r_md.loc[diff, ["home_team", "home_nickname"]].add_prefix("r_")],
    axis=1,
).drop_duplicates(subset=["home_team", "home_nickname", "r_home_nickname"]).head(10)

109 rows differ


,season,home_team,home_nickname,r_home_team,r_home_nickname
5,2016,Footscray,NaN,Footscray,NaN


In [48]:
print("Python blanks:", sorted(py.loc[py["home_nickname"].isna(), "home_team"].unique()))
print("R blanks:     ", sorted(r_md.loc[r_md["home_nickname"].isna(), "home_team"].unique()))

Python blanks: ['Footscray']
R blanks:      ['Footscray']


In [49]:
pd.testing.assert_frame_equal(py, r_md, check_dtype=False)